# Medical Image Denoising & Reconstruction — Brain Tumor MRI (IMPROVED)

> **Problem:** Train an autoencoder to clean noisy medical scans  
> **Target:** MRI / CT scan denoising related to Brain Tumor  
> **Techniques:** Convolutional Autoencoders with Skip Connections (U-Net style)  
> **Advanced:** Evaluate using PSNR, SSIM across multiple noise types (not just loss)

---

### What This Improved Version Fixes

| Problem in Old Version | Fix in This Version |
|----------------------|---------------------|
| Model has no idea which noise type it sees | **Noise type conditioning** — model receives noise type as input |
| No residual learning — tries to reconstruct full image | **Residual learning** — model learns NOISE, not image |
| S&P noise gets blurred by convolutions | **Median filter preprocessing** for S&P before U-Net |
| Random val noise makes tracking impossible | **Per-noise-type evaluation** every epoch |
| MSE-dominant loss = blurry outputs | **Rebalanced loss**: SSIM-primary + Edge-heavy |
| No curriculum — all noise at once from epoch 1 | **3-phase curriculum**: Gaussian -> Rician/Poisson -> S&P/Mixed |
| Plain U-Net blocks | **ResUNet + Channel Attention (SE blocks)** |
| BatchNorm in small batches causes instability | **GroupNorm** (batch-size independent) |
| PSNR/SSIM computed on CPU = slow | **GPU-accelerated PSNR/SSIM** during validation |

### Quick Start
1. **Add the dataset**: Click "+ Add Input" -> search "brain tumor mri dataset" -> Add
2. **Enable GPU**: Settings -> Accelerator -> GPU T4 x2 (or P100)
3. Run all cells in order
4. Training takes ~3.5-4 hours on T4 (Kaggle gives you 9h/week on GPU!)


---
## Step 1: Installation & Environment Setup

In [8]:
!pip install -q wandb scikit-image kagglehub

import torch
import os

# ─── wandb login via Kaggle Secrets ───
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb_api_key")
wandb.login(key=wandb_key)

# ─── Environment Check ───
print("=" * 55)
print("  ENVIRONMENT CHECK")
print("=" * 55)
print(f"  PyTorch:    {torch.__version__}")
print(f"  CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:        {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if torch.cuda.device_count() > 1:
        print(f"  GPU Count:  {torch.cuda.device_count()}")
print(f"  AMP:        {'Available' if hasattr(torch.amp, 'autocast') else 'Not available'}")
print("=" * 55)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


  ENVIRONMENT CHECK
  PyTorch:    2.10.0+cpu
  CUDA:       False
  AMP:        Available


In [7]:
!ls /kaggle/input/datasets/camorednex/

brain-mri-best-model  brain-mri-epoch30-checkpoint


---
## Step 2: Imports & Configuration

In [9]:
import os, glob, random, time, json, shutil
import numpy as np
from PIL import Image
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim

# ─────────────────────────────────────────────
# CONFIGURATION — All in One Place
# ─────────────────────────────────────────────
CONFIG = {
    # Data
    "image_size": 256,
    "batch_size": 8,
    "num_workers": 0,

    # Noise
    "noise_factor_max": 0.20,

    # Model — IMPROVED
    "architecture": "ResUNet + Attention + Noise Conditioning",
    "in_channels": 1,
    "num_noise_types": 5,
    "features": [64, 128, 256, 512],
    "dropout": 0.05,
    "use_residual": True,
    "use_attention": True,

    # Training — STABILIZED
    "epochs": 100,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 8,
    "scheduler": "cosine_warmup",
    "early_stopping_patience": 999,
    "use_amp": False,
    "grad_clip_norm": 0.5,
    "loss_cap": 0.5,

    # Resume from epoch 30 checkpoint
    "resume_from": "/kaggle/input/datasets/camorednex/brain-mri-epoch30-checkpoint/checkpoint_epoch_30.pth",

    # Curriculum — ORIGINAL (same as before)
    "curriculum": {
        "phase1_end": 30,
        "phase2_end": 60,
        "phase3_end": 100,
    },

    # Loss — STABILIZED (weights sum to 1.0)
    "loss_type": "combined_v2",
    "mse_w": 0.4,
    "ssim_w": 0.15,
    "mae_w": 0.3,
    "edge_w": 0.15,

    # Splits
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,

    # Output — Kaggle paths
    "checkpoint_dir": "/kaggle/working/",
    "local_backup_dir": "/kaggle/working/",
    "dataset": "Brain Tumor MRI (Kaggle)",
    "optimizer": "AdamW",
}

RESUME_FROM_CHECKPOINT = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["local_backup_dir"], exist_ok=True)
print(f"Checkpoint dir: {CONFIG['checkpoint_dir']}")
print(f"Architecture:   {CONFIG['architecture']}")
print(f"Residual learning: {CONFIG['use_residual']}")
print(f"AMP: {CONFIG['use_amp']} | Grad clip: {CONFIG['grad_clip_norm']} | Loss cap: {CONFIG['loss_cap']}")
print(f"Loss weights: Charb={CONFIG['mse_w']}, SSIM={CONFIG['ssim_w']}, L1={CONFIG['mae_w']}, Edge={CONFIG['edge_w']}")
print(f"Curriculum: Phase1={CONFIG['curriculum']['phase1_end']}, Phase2={CONFIG['curriculum']['phase2_end']}, Phase3={CONFIG['curriculum']['phase3_end']}")
print(f"Resume from: {CONFIG.get('resume_from', 'None')}")
print(f"Early stopping patience: {CONFIG['early_stopping_patience']}")

Device: cpu
Checkpoint dir: /kaggle/working/
Architecture:   ResUNet + Attention + Noise Conditioning
Residual learning: True
AMP: False | Grad clip: 0.5 | Loss cap: 0.5
Loss weights: Charb=0.4, SSIM=0.15, L1=0.3, Edge=0.15
Curriculum: Phase1=30, Phase2=60, Phase3=100
Resume from: /kaggle/input/datasets/camorednex/brain-mri-epoch30-checkpoint/checkpoint_epoch_30.pth
Early stopping patience: 999


---
## Step 3: Download Brain Tumor MRI Dataset

In [10]:
# ═══════════════════════════════════════════════
# Load Brain Tumor MRI Dataset from Kaggle Input
# ═══════════════════════════════════════════════

# On Kaggle, add the dataset to your notebook first:
#   Click '+ Add Input' on the right panel -> search 'brain tumor mri' -> Add
# The dataset will appear under /kaggle/input/

# Auto-detect the dataset path
dataset_path = None
for d in sorted(os.listdir('/kaggle/input')):
    candidate = os.path.join('/kaggle/input', d)
    if os.path.isdir(candidate):
        # Check for expected subdirs (Training/Testing or class folders)
        subdirs = []
        try:
            subdirs = os.listdir(candidate)
        except:
            continue
        if any(s.lower() in ['glioma', 'meningioma', 'notumor', 'pituitary', 'training', 'testing'] for s in subdirs):
            dataset_path = candidate
            break

if dataset_path is None:
    # Fallback: try common Kaggle dataset slug patterns
    possible_paths = [
        '/kaggle/input/brain-tumor-mri-dataset',
        '/kaggle/input/masoudnickparvar-brain-tumor-mri-dataset',
    ]
    for p in possible_paths:
        if os.path.exists(p):
            dataset_path = p
            break

if dataset_path is None:
    # Last resort: use first available input directory with images
    for d in sorted(os.listdir('/kaggle/input')):
        candidate = os.path.join('/kaggle/input', d)
        if os.path.isdir(candidate):
            dataset_path = candidate
            break

assert dataset_path is not None, 'No dataset found! Add the brain-tumor-mri-dataset via + Add Input'
print(f'Dataset path: {dataset_path}')
print(f'Contents: {os.listdir(dataset_path)}')

all_image_paths = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    all_image_paths.extend(glob.glob(os.path.join(dataset_path, '**', ext), recursive=True))

random.shuffle(all_image_paths)
print(f'Total images found: {len(all_image_paths)}')

class_counts = Counter()
for p in all_image_paths:
    for part in p.split('/'):
        if part.lower() in ['glioma', 'meningioma', 'notumor', 'pituitary']:
            class_counts[part.lower()] += 1
            break

print('\nClass distribution:')
for cls, cnt in sorted(class_counts.items()):
    print(f'  {cls}: {cnt}')

assert len(all_image_paths) > 0, f'No images found in {dataset_path}! Check the dataset was added correctly.'


Dataset path: /kaggle/input/datasets
Contents: ['masoudnickparvar', 'camorednex']
Total images found: 7200

Class distribution:
  glioma: 1800
  meningioma: 1800
  notumor: 1800
  pituitary: 1800


---
## Step 4: Noise Generation — 5 Types with Median Preprocessing for S&P

**Key improvement**: Salt & Pepper noise requires a fundamentally different approach than additive noise.
Convolutions blur S&P outliers — they don't remove them. We add a **median filter preprocessing** step
specifically for S&P noise that runs BEFORE the U-Net.

In [11]:
# ═══════════════════════════════════════════════
# 5 Noise Models for Medical Imaging
# ═══════════════════════════════════════════════

NOISE_TYPE_MAP = {'gaussian': 0, 'rician': 1, 'salt_pepper': 2, 'poisson': 3, 'mixed': 4}
NOISE_TYPE_NAMES = ['gaussian', 'rician', 'salt_pepper', 'poisson', 'mixed']

def add_gaussian_noise(image, std=0.1):
    return torch.clamp(image + torch.randn_like(image) * std, 0.0, 1.0)

def add_salt_pepper_noise(image, prob=0.02):
    noisy = image.clone()
    salt = torch.rand_like(image) < prob / 2
    pepper = torch.rand_like(image) < prob / 2
    noisy[salt] = 1.0
    noisy[pepper] = 0.0
    return noisy

def add_rician_noise(image, std=0.05):
    n1 = torch.randn_like(image) * std
    n2 = torch.randn_like(image) * std
    return torch.clamp(torch.sqrt((image + n1)**2 + n2**2), 0.0, 1.0)

def add_poisson_noise(image, scale=50.0):
    scaled = image * scale
    noisy = torch.poisson(scaled) / scale
    return torch.clamp(noisy, 0.0, 1.0)

def add_mixed_noise(image, gaussian_std=0.05, sp_prob=0.01):
    image = add_gaussian_noise(image, gaussian_std)
    image = add_salt_pepper_noise(image, sp_prob)
    return image

NOISE_FNS = {
    'gaussian': add_gaussian_noise,
    'rician': add_rician_noise,
    'salt_pepper': add_salt_pepper_noise,
    'poisson': add_poisson_noise,
    'mixed': add_mixed_noise,
}

# ─── Median Filter for S&P Preprocessing ───
# This is a GPU-friendly approximate median filter using
# a 3x3 morphological approach (dilate + erode average)
def median_filter_approx(image, kernel_size=3):
    """Approximate median filter using morphological operations.
    Exact median on GPU is expensive; this approximation works
    well for removing S&P impulse noise outliers.
    """
    pad = kernel_size // 2
    # Collect shifted versions and take the middle value
    shifts = []
    for di in range(-pad, pad+1):
        for dj in range(-pad, pad+1):
            shifts.append(F.pad(image, [pad, pad, pad, pad], mode='reflect')
                       [:, :, di:di+image.shape[2], dj:dj+image.shape[3]])
    stacked = torch.stack(shifts, dim=0)
    # Sort and take median (middle value)
    result = stacked.median(dim=0).values
    return result


# ─── Visualize noise types ───
sample_img = Image.open(all_image_paths[0]).convert("L")
sample_tensor = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])(sample_img)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
vis_items = [
    ('Original', None, None),
    ('Gaussian (0.10)', 'gaussian', 0.10),
    ('Rician (0.08)', 'rician', 0.08),
    ('S&P (2%)', 'salt_pepper', 0.02),
    ('Poisson (30)', 'poisson', 30.0),
    ('Mixed', 'mixed', 0.05),
]
for ax, (label, nt, level) in zip(axes.flatten(), vis_items):
    if nt is None:
        img_np = sample_tensor.squeeze().numpy()
    else:
        img_np = NOISE_FNS[nt](sample_tensor, level).squeeze().numpy()
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Noise Types in Medical Imaging', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("5 noise types ready")

5 noise types ready


---
## Step 5: Dataset with Curriculum Noise & Noise Type Labels

**Key improvement**: The dataset now returns the **noise type index** alongside the noisy/clean pair.
This allows the model to condition on the noise type. Also, curriculum training changes which noise
types are active based on the current epoch.

In [6]:
class BrainMRIDatasetV2(Dataset):
    """Improved dataset that returns (noisy, clean, noise_type_idx) for conditioning.
    
    Curriculum mode: noise_types parameter can be changed between epochs
    to implement curriculum training.
    """
    def __init__(self, image_paths, image_size=256, noise_factor_max=0.20,
                 augment=False, noise_types=None):
        self.image_paths = image_paths
        self.image_size = image_size
        self.noise_factor_max = noise_factor_max
        self.augment = augment
        # Active noise types — can be updated for curriculum
        self.noise_types = noise_types or ['gaussian']

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.1),
                transforms.ToTensor(),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
            ])

    def set_noise_types(self, noise_types):
        """Update active noise types for curriculum training."""
        self.noise_types = noise_types

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        clean = self.transform(img)

        # Select noise type and level
        noise_type = random.choice(self.noise_types)
        noise_fn = NOISE_FNS[noise_type]
        noise_idx = NOISE_TYPE_MAP[noise_type]

        # Random noise level within range
        level = np.random.uniform(0.02, self.noise_factor_max)
        noisy = noise_fn(clean, level)

        return noisy, clean, noise_idx


class BrainMRIEvalDataset(Dataset):
    """Evaluation dataset — FIXED noise type per sample for fair per-type tracking."""
    def __init__(self, image_paths, image_size=256, noise_type='gaussian',
                 noise_level=0.10):
        self.image_paths = image_paths
        self.image_size = image_size
        self.noise_type = noise_type
        self.noise_level = noise_level
        self.noise_idx = NOISE_TYPE_MAP[noise_type]
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        clean = self.transform(img)
        noisy = NOISE_FNS[self.noise_type](clean, self.noise_level)
        return noisy, clean, self.noise_idx


# ─── Split dataset ───
n = len(all_image_paths)
n_train = int(n * CONFIG["train_split"])
n_val = int(n * CONFIG["val_split"])

train_paths = all_image_paths[:n_train]
val_paths = all_image_paths[n_train:n_train + n_val]
test_paths = all_image_paths[n_train + n_val:]

# Training dataset — starts with Gaussian only (curriculum)
train_ds = BrainMRIDatasetV2(
    train_paths, CONFIG["image_size"],
    noise_factor_max=CONFIG["noise_factor_max"],
    augment=True, noise_types=['gaussian']  # Will be updated by curriculum
)

# Per-noise-type validation datasets for tracking
val_datasets = {}
val_loaders = {}
for nt in NOISE_TYPE_NAMES:
    level = 0.10 if nt not in ['poisson'] else 50.0
    level = 0.02 if nt == 'salt_pepper' else level
    ds = BrainMRIEvalDataset(val_paths, CONFIG["image_size"], nt, level)
    val_datasets[nt] = ds
    val_loaders[nt] = DataLoader(ds, batch_size=CONFIG["batch_size"],
                                shuffle=False, num_workers=CONFIG["num_workers"],
                                pin_memory=True)

# Also a combined val loader for quick overall check
val_ds_combined = BrainMRIDatasetV2(
    val_paths, CONFIG["image_size"],
    noise_factor_max=CONFIG["noise_factor_max"],
    augment=False, noise_types=NOISE_TYPE_NAMES
)
val_loader_combined = DataLoader(val_ds_combined, batch_size=CONFIG["batch_size"],
                                  shuffle=False, num_workers=CONFIG["num_workers"],
                                  pin_memory=True)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=CONFIG["num_workers"], pin_memory=True,
                          drop_last=True)

print(f"Train: {len(train_ds)} | Val per type: {len(val_datasets['gaussian'])}")
print(f"Curriculum: starting with Gaussian only")
print(f"Validation: {len(val_datasets)} separate loaders (one per noise type)")

Train: 5760 | Val per type: 720
Curriculum: starting with Gaussian only
Validation: 5 separate loaders (one per noise type)


---
## Step 6: Improved Architecture — ResUNet + Attention + Noise Conditioning

### Key Improvements Over Original:
1. **Residual learning**: Model outputs NOISE (which is simpler), then subtracts from input -> clean image
2. **Noise type embedding**: Model receives a learned embedding of the noise type, so it can specialize
3. **SE (Squeeze-Excitation) attention**: Channel attention lets the model focus on relevant features
4. **GroupNorm**: Stable regardless of batch size (BatchNorm unstable with batch_size=8)
5. **Residual connections in DoubleConv**: Skip connections within each block for better gradient flow
6. **Median filter branch for S&P**: Special preprocessing for impulse noise

In [12]:
# ═══════════════════════════════════════════════
# ResUNet + Attention + Noise Conditioning
# ═══════════════════════════════════════════════

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 8), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels // reduction, 8), channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.squeeze(x).view(b, c)
        y = self.excite(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ResDoubleConv(nn.Module):
    """Double Conv with residual connection + SE attention."""
    def __init__(self, in_ch, out_ch, dropout=0.05, use_attention=True):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
        )
        self.se = SEBlock(out_ch) if use_attention else nn.Identity()
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.se(out)
        return out + residual


class NoiseConditioning(nn.Module):
    """Learned embedding for noise type."""
    def __init__(self, num_noise_types=5, embed_dim=16, spatial_size=256):
        super().__init__()
        self.embedding = nn.Embedding(num_noise_types, embed_dim)
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.spatial_size = spatial_size

    def forward(self, noise_idx, batch_size):
        emb = self.embedding(noise_idx)
        emb = self.proj(emb)
        emb = emb.unsqueeze(-1).unsqueeze(-1)
        emb = emb.expand(-1, -1, self.spatial_size, self.spatial_size)
        return emb


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.05, use_attention=True):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = ResDoubleConv(in_ch, out_ch, dropout, use_attention)

    def forward(self, x):
        skip = self.conv(x)
        pooled = self.pool(skip)
        return skip, pooled


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, dropout=0.05, use_attention=True):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ResDoubleConv(in_ch // 2 + skip_ch, out_ch, dropout, use_attention)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class ResUNetDenoiser(nn.Module):
    """ResUNet with Noise Conditioning and Channel Attention."""
    def __init__(self, in_channels=1, num_noise_types=5, noise_embed_dim=16,
                 features=[64, 128, 256, 512], dropout=0.05,
                 use_residual=True, use_attention=True, spatial_size=256):
        super().__init__()
        self.use_residual = use_residual
        self.noise_cond = NoiseConditioning(num_noise_types, noise_embed_dim, spatial_size)

        # First conv takes image + noise embedding (ResDoubleConv, NOT EncoderBlock)
        self.enc0 = ResDoubleConv(in_channels + noise_embed_dim, features[0], dropout, use_attention)
        # EncoderBlocks return (skip, pooled) tuples
        self.enc1 = EncoderBlock(features[0], features[1], dropout, use_attention)
        self.enc2 = EncoderBlock(features[1], features[2], dropout, use_attention)
        self.enc3 = EncoderBlock(features[2], features[3], dropout, use_attention)

        self.bottleneck = ResDoubleConv(features[3], features[3] * 2, dropout, use_attention)

        self.dec3 = DecoderBlock(features[3] * 2, features[3], features[3], dropout, use_attention)
        self.dec2 = DecoderBlock(features[3], features[2], features[2], dropout, use_attention)
        self.dec1 = DecoderBlock(features[2], features[1], features[1], dropout, use_attention)
        self.dec0 = DecoderBlock(features[1], features[0], features[0], dropout, use_attention)

        self.out_conv = nn.Conv2d(features[0], in_channels, kernel_size=1)

    def forward(self, x, noise_idx):
        noise_emb = self.noise_cond(noise_idx, x.shape[0])
        x_cond = torch.cat([x, noise_emb], dim=1)

        # enc0 is ResDoubleConv — returns ONE tensor, pool manually
        s0 = self.enc0(x_cond)
        p0 = F.max_pool2d(s0, 2)

        # enc1-3 are EncoderBlocks — return (skip, pooled) tuples
        s1, p1 = self.enc1(p0)
        s2, p2 = self.enc2(p1)
        s3, p3 = self.enc3(p2)

        b = self.bottleneck(p3)

        d3 = self.dec3(b, s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)
        d0 = self.dec0(d1, s0)

        noise_est = self.out_conv(d0)

        if self.use_residual:
            denoised = x - noise_est
            return torch.clamp(denoised, 0.0, 1.0)
        else:
            return torch.sigmoid(noise_est)


# ─── Create model ───
model = ResUNetDenoiser(
    in_channels=CONFIG["in_channels"],
    num_noise_types=CONFIG["num_noise_types"],
    noise_embed_dim=16,
    features=CONFIG["features"],
    dropout=CONFIG["dropout"],
    use_residual=CONFIG["use_residual"],
    use_attention=CONFIG["use_attention"],
    spatial_size=CONFIG["image_size"],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ResUNetDenoiser | Parameters: {total_params:,}")
print(f"Residual learning: {CONFIG['use_residual']}")
print(f"Noise conditioning: {CONFIG['num_noise_types']} types")
print(f"Channel attention (SE): {CONFIG['use_attention']}")

# Quick shape test
test_input = torch.randn(2, 1, 256, 256).to(DEVICE)
test_noise_idx = torch.tensor([0, 2]).to(DEVICE)
test_output = model(test_input, test_noise_idx)
print(f"Input: {test_input.shape} -> Output: {test_output.shape}")
del test_input, test_noise_idx, test_output

ResUNetDenoiser | Parameters: 32,662,657
Residual learning: True
Noise conditioning: 5 types
Channel attention (SE): True
Input: torch.Size([2, 1, 256, 256]) -> Output: torch.Size([2, 1, 256, 256])


---
## Step 7: Rebalanced Loss Function

Old version: `L = 0.5*MSE + 0.3*SSIM + 0.1*MAE + 0.1*Edge`

Problem: MSE dominates and produces **blurry** outputs. For S&P noise, MSE penalizes
any deviation from 0 or 1 — which is exactly what we need to do!

New version: `L = 0.2*MSE + 0.4*SSIM + 0.15*MAE + 0.25*Edge`
- SSIM is now primary (preserves structure)
- Edge weight doubled (critical for tumor boundaries)
- Charbonnier loss instead of MSE (more robust to outliers = better for S&P)

In [8]:
# ═══════════════════════════════════════════════
# Rebalanced Combined Loss
# ═══════════════════════════════════════════════

def gaussian_kernel(kernel_size=11, sigma=1.5):
    coords = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g /= g.sum()
    return g[:, None] * g[None, :]


class SSIMLoss(nn.Module):
    """SSIM-based loss = 1 - SSIM"""
    def __init__(self, kernel_size=11, sigma=1.5, channels=1):
        super().__init__()
        kernel = gaussian_kernel(kernel_size, sigma)
        kernel = kernel.unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)
        self.register_buffer('kernel', kernel)
        self.channels = channels
        self.padding = kernel_size // 2

    def forward(self, pred, target):
        C1, C2 = 0.01**2, 0.03**2
        mu_x = F.conv2d(pred, self.kernel, padding=self.padding, groups=self.channels)
        mu_y = F.conv2d(target, self.kernel, padding=self.padding, groups=self.channels)
        mu_x_sq, mu_y_sq, mu_xy = mu_x**2, mu_y**2, mu_x * mu_y
        sigma_x = F.conv2d(pred**2, self.kernel, padding=self.padding, groups=self.channels) - mu_x_sq
        sigma_y = F.conv2d(target**2, self.kernel, padding=self.padding, groups=self.channels) - mu_y_sq
        sigma_xy = F.conv2d(pred*target, self.kernel, padding=self.padding, groups=self.channels) - mu_xy
        num = (2 * mu_xy + C1) * (2 * sigma_xy + C2)
        den = (mu_x_sq + mu_y_sq + C1) * (sigma_x + sigma_y + C2)
        return 1.0 - (num / (den + 1e-8)).mean()


class CharbonnierLoss(nn.Module):
    """Charbonnier (L1-L2 hybrid) loss — more robust to outliers than MSE.
    Crucial for S&P noise where some pixels are COMPLETELY wrong.
    L1 tends to ignore outliers; MSE is too sensitive; Charbonnier is the sweet spot.
    """
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff**2 + self.eps**2))


class EdgeLoss(nn.Module):
    """Edge-preserving loss using Sobel gradients."""
    def __init__(self):
        super().__init__()
        sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)
        sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)
        self.register_buffer('sobel_x', sx.unsqueeze(0).unsqueeze(0))
        self.register_buffer('sobel_y', sy.unsqueeze(0).unsqueeze(0))

    def forward(self, pred, target):
        def grad(x):
            return torch.sqrt(F.conv2d(x, self.sobel_x, padding=1)**2 +
                              F.conv2d(x, self.sobel_y, padding=1)**2 + 1e-8)
        return F.l1_loss(grad(pred), grad(target))


class CombinedLossV2(nn.Module):
    """Rebalanced: L = 0.2*Charbonnier + 0.4*SSIM + 0.15*L1 + 0.25*Edge"""
    def __init__(self, mse_w=0.2, ssim_w=0.4, mae_w=0.15, edge_w=0.25, channels=1):
        super().__init__()
        self.mse_w, self.ssim_w, self.mae_w, self.edge_w = mse_w, ssim_w, mae_w, edge_w
        self.charb = CharbonnierLoss()
        self.mae = nn.L1Loss()
        self.ssim = SSIMLoss(channels=channels)
        self.edge = EdgeLoss()

    def forward(self, pred, target):
        losses, total = {}, 0.0
        if self.mse_w > 0:
            losses['charb'] = self.charb(pred, target); total += self.mse_w * losses['charb']
        if self.ssim_w > 0:
            losses['ssim'] = self.ssim(pred, target); total += self.ssim_w * losses['ssim']
        if self.mae_w > 0:
            losses['mae'] = self.mae(pred, target); total += self.mae_w * losses['mae']
        if self.edge_w > 0:
            losses['edge'] = self.edge(pred, target); total += self.edge_w * losses['edge']
        losses['total'] = total
        return total, losses


criterion = CombinedLossV2(
    mse_w=CONFIG["mse_w"], ssim_w=CONFIG["ssim_w"],
    mae_w=CONFIG["mae_w"], edge_w=CONFIG["edge_w"],
    channels=CONFIG["in_channels"]
).to(DEVICE)

print(f"Loss: CombinedV2 (Charbonnier + SSIM + L1 + Edge)")
print(f"  Weights: Charb={CONFIG['mse_w']}, SSIM={CONFIG['ssim_w']}, L1={CONFIG['mae_w']}, Edge={CONFIG['edge_w']}")

# Quick test
t1, t2 = torch.rand(2, 1, 256, 256).to(DEVICE), torch.rand(2, 1, 256, 256).to(DEVICE)
total, bd = criterion(t1, t2)
print(f"  Test: Total={total:.4f} | " + " | ".join(f"{k}: {v:.4f}" for k, v in bd.items() if k != 'total'))
del t1, t2

Loss: CombinedV2 (Charbonnier + SSIM + L1 + Edge)
  Weights: Charb=0.4, SSIM=0.15, L1=0.3, Edge=0.15
  Test: Total=0.4866 | charb: 0.3328 | ssim: 0.9773 | mae: 0.3328 | edge: 0.7135


---
## Step 8: GPU-Accelerated Metrics & Per-Noise-Type Evaluation

Computing PSNR/SSIM on CPU (skimage) is SLOW. We implement GPU versions
for the training loop and only use skimage for final evaluation.

In [9]:
# ═══════════════════════════════════════════════
# GPU-Accelerated Metrics
# ═══════════════════════════════════════════════

def gpu_psnr(pred, target, data_range=1.0):
    """GPU PSNR — no CPU transfer needed."""
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=[1,2,3])
    mse = torch.clamp(mse, min=1e-10)
    psnr = 10.0 * torch.log10(data_range**2 / mse)
    return psnr.mean().item()

def gpu_ssim(pred, target, data_range=1.0):
    """GPU SSIM using the same kernel as SSIMLoss (shared buffer)."""
    C1, C2 = 0.01**2, 0.03**2
    kernel = criterion.ssim.kernel   # Reuse the kernel buffer
    pad = criterion.ssim.padding
    mu_x = F.conv2d(pred, kernel, padding=pad, groups=1)
    mu_y = F.conv2d(target, kernel, padding=pad, groups=1)
    mu_x_sq, mu_y_sq, mu_xy = mu_x**2, mu_y**2, mu_x*mu_y
    sigma_x = F.conv2d(pred**2, kernel, padding=pad, groups=1) - mu_x_sq
    sigma_y = F.conv2d(target**2, kernel, padding=pad, groups=1) - mu_y_sq
    sigma_xy = F.conv2d(pred*target, kernel, padding=pad, groups=1) - mu_xy
    num = (2*mu_xy + C1) * (2*sigma_xy + C2)
    den = (mu_x_sq + mu_y_sq + C1) * (sigma_x + sigma_y + C2)
    return (num / (den + 1e-8)).mean().item()

@torch.no_grad()
def evaluate_per_noise_type(model, val_loaders, criterion, device, use_amp=False):
    """Evaluate PSNR/SSIM for EACH noise type separately.
    This is the KEY improvement — you can now see exactly which
    noise types the model is struggling with.
    """
    model.eval()
    results = {}
    overall_psnr, overall_ssim, overall_loss = [], [], []

    for noise_type, loader in val_loaders.items():
        psnr_vals, ssim_vals, loss_vals = [], [], []
        for noisy, clean, noise_idx in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            noise_idx = noise_idx.to(device)

            if use_amp:
                with torch.amp.autocast('cuda'):
                    output = model(noisy, noise_idx)
                    loss, _ = criterion(output, clean)
            else:
                output = model(noisy, noise_idx)
                loss, _ = criterion(output, clean)

            loss_vals.append(loss.item())
            psnr_vals.append(gpu_psnr(output, clean))
            ssim_vals.append(gpu_ssim(output, clean))

        avg_psnr = np.mean(psnr_vals)
        avg_ssim = np.mean(ssim_vals)
        avg_loss = np.mean(loss_vals)
        results[noise_type] = {'psnr': avg_psnr, 'ssim': avg_ssim, 'loss': avg_loss}
        overall_psnr.append(avg_psnr)
        overall_ssim.append(avg_ssim)
        overall_loss.append(avg_loss)

    results['overall'] = {
        'psnr': np.mean(overall_psnr),
        'ssim': np.mean(overall_ssim),
        'loss': np.mean(overall_loss),
    }
    return results

print("GPU metrics and per-noise-type evaluation ready")

GPU metrics and per-noise-type evaluation ready


---
## Step 9: Training Loop with Curriculum + Per-Type Tracking

### Curriculum Strategy:
- **Phase 1 (Epochs 1-30)**: Gaussian only — builds a strong base denoiser
- **Phase 2 (Epochs 31-60)**: + Rician + Poisson — adds MRI-specific noise types
- **Phase 3 (Epochs 61-100)**: All 5 types — full robustness including S&P and Mixed

### Why This Works:
Starting with all 5 noise types from epoch 1 is like trying to learn 5 languages simultaneously.
Curriculum training lets the model first master Gaussian (easiest, most common), then
gradually add harder noise types.

In [10]:
# ═══════════════════════════════════════════════
# Training Loop with Curriculum + Per-Type Eval
# STABILIZED: No AMP, No Scaler, NaN Detection,
#   Loss Capping, Tighter Gradient Clipping, wandb
# ═══════════════════════════════════════════════

def get_curriculum_noise_types(epoch):
    """Return active noise types based on curriculum phase."""
    cur = CONFIG["curriculum"]
    if epoch <= cur["phase1_end"]:
        return ['gaussian']
    elif epoch <= cur["phase2_end"]:
        return ['gaussian', 'rician', 'poisson']
    else:
        return NOISE_TYPE_NAMES  # all 5


def save_checkpoint(state, filename, drive_dir, local_dir):
    local_path = os.path.join(local_dir, filename)
    torch.save(state, local_path)
    drive_path = os.path.join(drive_dir, filename)
    try:
        if local_path != drive_path:
            shutil.copy2(local_path, drive_path)
    except Exception as e:
        print(f"  Save error: {e}")
    return local_path


# ─── Resume from checkpoint ───
start_epoch = 1
best_val_psnr = 0.0
patience_counter = 0
history = {
    'train_loss': [], 'val_loss': [],
    'val_psnr': [], 'val_ssim': [],
    'per_noise_psnr': {nt: [] for nt in NOISE_TYPE_NAMES},
    'per_noise_ssim': {nt: [] for nt in NOISE_TYPE_NAMES},
    'curriculum_phase': [],
    'lr': [],
}

# ← CHANGED: Resume from CONFIG["resume_from"] instead of RESUME_FROM_CHECKPOINT
if CONFIG.get("resume_from") and os.path.exists(CONFIG["resume_from"]):
    print(f"\n Resuming from {CONFIG['resume_from']}")
    ckpt = torch.load(CONFIG["resume_from"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_psnr = ckpt.get("val_psnr", 0.0)
    history = ckpt.get("history", history)
    print(f" Resumed from epoch {ckpt.get('epoch', 0)}, starting at epoch {start_epoch}")

# ─── Optimizer & Scheduler ───
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

# ← CHANGED: Load optimizer from same resume checkpoint
if CONFIG.get("resume_from") and os.path.exists(CONFIG["resume_from"]):
    ckpt = torch.load(CONFIG["resume_from"], map_location=DEVICE, weights_only=False)
    if "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        print(" Optimizer state loaded from checkpoint")

# Cosine annealing with warmup
def get_lr_lambda(epoch):
    warmup = CONFIG["warmup_epochs"]
    total = CONFIG["epochs"]
    if epoch < warmup:
        return (epoch + 1) / warmup   # Linear warmup
    progress = (epoch - warmup) / (total - warmup)
    return 0.5 * (1 + np.cos(np.pi * progress))  # Cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr_lambda)

# ← CHANGED: Advance scheduler to match start_epoch
if start_epoch > 1:
    for _ in range(start_epoch - 1):
        scheduler.step()
    print(f" Scheduler advanced to epoch {start_epoch}")

# ─── NO AMP, NO SCALER ───
use_amp = False

# ← CHANGED: wandb init with correct run ID
wandb_run = wandb.init(
    project="medical-denoising-brain-mri",
    entity="camorednex-_",
    name=f"resunet-stable-lr{CONFIG['learning_rate']}-ssim{CONFIG['ssim_w']}",
    config=CONFIG,
    resume="allow",
    id="twqu38pi"
)

# ─── TRAINING ───
print(f"\n{'='*70}")
print(f"  TRAINING: {CONFIG['architecture']}")
print(f"  Epochs: {CONFIG['epochs']} | Batch: {CONFIG['batch_size']}")
print(f"  Resuming from epoch {start_epoch}")  # ← CHANGED
print(f"  Curriculum: Phase1(G) -> Phase2(G+R+P) -> Phase3(All5)")
print(f"  Residual learning: {CONFIG['use_residual']}")
print(f"  AMP: {use_amp} | Grad clip: {CONFIG['grad_clip_norm']} | Loss cap: {CONFIG['loss_cap']}")
print(f"  Loss: Charb={CONFIG['mse_w']} + SSIM={CONFIG['ssim_w']} + L1={CONFIG['mae_w']} + Edge={CONFIG['edge_w']}")
print(f"  Early stopping patience: {CONFIG['early_stopping_patience']}")  # ← CHANGED
print(f"{'='*70}\n")

nan_count = 0
skip_count = 0

for epoch in range(start_epoch, CONFIG["epochs"] + 1):  # ← already uses start_epoch
    t0 = time.time()

    # ─── Update curriculum ───
    active_noise = get_curriculum_noise_types(epoch)
    train_ds.set_noise_types(active_noise)
    phase = 1 if epoch <= CONFIG["curriculum"]["phase1_end"] else \
            2 if epoch <= CONFIG["curriculum"]["phase2_end"] else 3

    # ─── Train ───
    model.train()
    train_loss = 0.0
    batch_count = 0
    pbar = tqdm(train_loader, desc=f"  E{epoch:03d}/{CONFIG['epochs']} P{phase} [{','.join(nt[:3] for nt in active_noise)}]",
                leave=False, ncols=120)
    for noisy, clean, noise_idx in pbar:
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        noise_idx = noise_idx.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        # ─── Forward pass (NO autocast) ───
        output = model(noisy, noise_idx)
        loss, loss_dict = criterion(output, clean)

        # ─── NaN/Inf detection ───
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            if nan_count <= 3:
                print(f"  ⚠ NaN/Inf loss at epoch {epoch}, batch {batch_count}. Skipping.")
            continue

        # ─── LOSS CAPPING — skip anomalous batches ───
        if loss.item() > CONFIG["loss_cap"]:
            skip_count += 1
            if skip_count <= 3:
                print(f"  ⚠ Capped loss {loss.item():.4f} > {CONFIG['loss_cap']} at epoch {epoch}, batch {batch_count}. Skipping.")
            continue

        # ─── Backward pass ───
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["grad_clip_norm"])
        optimizer.step()

        train_loss += loss.item()
        batch_count += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    if batch_count > 0:
        train_loss /= batch_count
    else:
        print(f"  ⚠ WARNING: All batches skipped in epoch {epoch}!")

    # ─── Validate per noise type (every 5 epochs for speed) ───
    if epoch % 5 == 0 or epoch == 1 or epoch == CONFIG["epochs"]:
        val_results = evaluate_per_noise_type(model, val_loaders, criterion, DEVICE, use_amp=False)
        val_psnr = val_results['overall']['psnr']
        val_ssim = val_results['overall']['ssim']
        val_loss = val_results['overall']['loss']

        # Print per-noise-type results
        print(f"\n  Epoch {epoch:03d} | Phase {phase} | Noise: {active_noise}")
        print(f"  {'Noise Type':>15} | {'PSNR':>8} | {'SSIM':>8}")
        print(f"  {'-'*15}-+-{'-'*8}-+-{'-'*8}")
        for nt in NOISE_TYPE_NAMES:
            if nt in val_results:
                print(f"  {nt:>15} | {val_results[nt]['psnr']:>7.2f} | {val_results[nt]['ssim']:>7.4f}")
        print(f"  {'OVERALL':>15} | {val_psnr:>7.2f} | {val_ssim:>7.4f}")
    else:
        # Quick combined validation for other epochs
        model.eval()
        val_loss_quick = 0.0
        psnr_q, ssim_q = [], []
        with torch.no_grad():
            for noisy, clean, noise_idx in val_loader_combined:
                noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
                noise_idx = noise_idx.to(DEVICE)
                output = model(noisy, noise_idx)
                loss, _ = criterion(output, clean)
                val_loss_quick += loss.item()
                psnr_q.append(gpu_psnr(output, clean))
                ssim_q.append(gpu_ssim(output, clean))
        val_loss = val_loss_quick / len(val_loader_combined)
        val_psnr = np.mean(psnr_q)
        val_ssim = np.mean(ssim_q)
        val_results = None

    # ─── Scheduler step ───
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]

    # ─── Record history ───
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_psnr'].append(val_psnr)
    history['val_ssim'].append(val_ssim)
    history['curriculum_phase'].append(phase)
    history['lr'].append(lr)

    if val_results:
        for nt in NOISE_TYPE_NAMES:
            if nt in val_results:
                history['per_noise_psnr'][nt].append(val_results[nt]['psnr'])
                history['per_noise_ssim'][nt].append(val_results[nt]['ssim'])

    # ─── wandb logging ───
    wandb_log = {
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_psnr': val_psnr,
        'val_ssim': val_ssim,
        'lr': lr,
        'phase': phase,
        'epoch': epoch,
    }
    if val_results:
        for nt in NOISE_TYPE_NAMES:
            if nt in val_results:
                wandb_log[f'psnr_{nt}'] = val_results[nt]['psnr']
                wandb_log[f'ssim_{nt}'] = val_results[nt]['ssim']
    wandb.log(wandb_log, step=epoch)

    # ─── Save best model (based on overall PSNR) ───
    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        patience_counter = 0
        state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_psnr': val_psnr,
            'val_ssim': val_ssim,
            'val_loss': val_loss,
            'history': history,
            'config': CONFIG,
        }
        saved = save_checkpoint(state, "best_model.pth",
                               CONFIG["checkpoint_dir"], CONFIG["local_backup_dir"])
        print(f"  *** Best model saved! PSNR: {val_psnr:.2f} ***")
    else:
        patience_counter += 1

    # ─── Periodic checkpoint every 10 epochs (backup) ───
    if epoch % 10 == 0:
        periodic_state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_psnr': val_psnr,
            'val_ssim': val_ssim,
            'val_loss': val_loss,
            'history': history,
            'config': CONFIG,
        }
        save_checkpoint(periodic_state, f"checkpoint_epoch_{epoch}.pth",
                       CONFIG["checkpoint_dir"], CONFIG["local_backup_dir"])

    elapsed = time.time() - t0
    print(f"  Epoch [{epoch:03d}/{CONFIG['epochs']}] P{phase} | "
          f"Train: {train_loss:.5f} | Val: {val_loss:.5f} | "
          f"PSNR: {val_psnr:.2f} | SSIM: {val_ssim:.4f} | "
          f"LR: {lr:.2e} | Time: {elapsed:.1f}s")

    # ─── Early stopping ───
    if patience_counter >= CONFIG["early_stopping_patience"]:
        print(f"\n  Early stopping at epoch {epoch} (patience={patience_counter})")
        break

# ─── Finalize wandb ───
wandb.finish()

print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE")
print(f"  Best PSNR: {best_val_psnr:.2f} dB")
print(f"  Total epochs: {epoch}")
print(f"  NaN batches skipped: {nan_count}")
print(f"  Capped batches skipped: {skip_count}")
print(f"{'='*70}")


 Resuming from /kaggle/input/datasets/camorednex/brain-mri-epoch30-checkpoint/checkpoint_epoch_30.pth
 Resumed from epoch 30, starting at epoch 31
 Optimizer state loaded from checkpoint
 Scheduler advanced to epoch 31


/tmp/ipykernel_23/1684411093.py:81: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
wandb: setting up run twqu38pi
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260516_145453-twqu38pi
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run resunet-stable-lr0.0005-ssim0.15
wandb: ⭐️ View project at https://wandb.ai/camorednex-_/medical-denoising-brain-mri
wandb: 🚀 View run at https://wandb.ai/camorednex-_/medical-denoising-brain-mri/runs/twqu38pi



  TRAINING: ResUNet + Attention + Noise Conditioning
  Epochs: 100 | Batch: 8
  Resuming from epoch 31
  Curriculum: Phase1(G) -> Phase2(G+R+P) -> Phase3(All5)
  Residual learning: True
  AMP: False | Grad clip: 0.5 | Loss cap: 0.5
  Loss: Charb=0.4 + SSIM=0.15 + L1=0.3 + Edge=0.15
  Early stopping patience: 999



  E031/100 P2 [gau,ric,poi]:   0%|                                                              | 0/720 [00:00…

  Epoch [031/100] P2 | Train: 0.08165 | Val: 0.14076 | PSNR: 23.66 | SSIM: 0.6354 | LR: 4.27e-04 | Time: 567.7s


...


  Epoch 100 | Phase 3 | Noise: ['gaussian', 'rician', 'salt_pepper', 'poisson', 'mixed']
       Noise Type |     PSNR |     SSIM
  ----------------+----------+---------
         gaussian |   32.79 |  0.8961
           rician |   32.34 |  0.8795
      salt_pepper |   52.65 |  0.9791
          poisson |   27.93 |  0.8734
            mixed |   32.64 |  0.8945
          OVERALL |   35.67 |  0.9045
  *** Best model saved! PSNR: 35.67 ***


wandb: updating run metadata


  Epoch [100/100] P3 | Train: 0.04644 | Val: 0.03280 | PSNR: 35.67 | SSIM: 0.9045 | LR: 0.00e+00 | Time: 648.3s


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
wandb:               lr ███▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
wandb:            phase ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁██████████████████████
wandb:    psnr_gaussian ▅▁▇▆▆▇▆▇▇█████
wandb:       psnr_mixed ▅▃▂▃▂▁████████
wandb:     psnr_poisson █▆▁▃▃▁▂▃▃▃▂▂▂▂
wandb:      psnr_rician ▅▁▆▆▇▇▇▇▇█████
wandb: psnr_salt_pepper ▁▁▁▁▁▁▇▇██████
wandb:    ssim_gaussian ▅▁▇▆▇▇▇█▇█████
wandb:       ssim_mixed ▅▂▂▃▂▁████████
wandb:               +7 ...
wandb: 
wandb: Run summary:
wandb:            epoch 100
wandb:               lr 0
wandb:            phase 3
wandb:    psnr_gaussian 32.78542
wandb:       psnr_mixed 32.6441
wandb:     psnr_poisson 27.93053
wandb:      psnr_rician 32.34344
wandb: psnr_salt_pepper 52.65084
wandb:    ssim_gaussian 0.89614
wandb:       ssim_mixed 0.89451
wandb:               +7 ...
wandb: 
wandb: 🚀 View run resun


  TRAINING COMPLETE
  Best PSNR: 35.67 dB
  Total epochs: 100
  NaN batches skipped: 0
  Capped batches skipped: 0


---
## Step 10: Training Curves & Per-Noise-Type Analysis

In [11]:
# ═══════════════════════════════════════════════
# Training Curves
# ═══════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss
axes[0,0].plot(history['train_loss'], label='Train', alpha=0.8)
axes[0,0].plot(history['val_loss'], label='Val')
axes[0,0].set_title('Loss', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Overall PSNR/SSIM
axes[0,1].plot(history['val_psnr'], color='green')
axes[0,1].set_title('Overall PSNR (dB)', fontweight='bold'); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(history['val_ssim'], color='blue')
axes[0,2].set_title('Overall SSIM', fontweight='bold'); axes[0,2].grid(True, alpha=0.3)

# Per-noise-type PSNR
colors = {'gaussian': 'red', 'rician': 'orange', 'salt_pepper': 'purple', 'poisson': 'brown', 'mixed': 'teal'}
for nt in NOISE_TYPE_NAMES:
    if history['per_noise_psnr'][nt]:
        axes[1,0].plot(history['per_noise_psnr'][nt], label=nt, color=colors[nt])
axes[1,0].set_title('PSNR per Noise Type', fontweight='bold')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# Per-noise-type SSIM
for nt in NOISE_TYPE_NAMES:
    if history['per_noise_ssim'][nt]:
        axes[1,1].plot(history['per_noise_ssim'][nt], label=nt, color=colors[nt])
axes[1,1].set_title('SSIM per Noise Type', fontweight='bold')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)

# Learning rate
axes[1,2].plot(history['lr'], color='gray')
axes[1,2].set_title('Learning Rate', fontweight='bold'); axes[1,2].grid(True, alpha=0.3)
# Mark curriculum phases
for ax in axes.flatten():
    for phase_epoch in [CONFIG['curriculum']['phase1_end'], CONFIG['curriculum']['phase2_end']]:
        if phase_epoch < len(history['train_loss']):
            ax.axvline(x=phase_epoch, color='red', linestyle='--', alpha=0.3)

plt.suptitle('Training History — ResUNet + Noise Conditioning + Curriculum', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["local_backup_dir"], "training_history.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Step 11: Comprehensive Evaluation — All Noise Types & Levels

In [12]:
# ═══════════════════════════════════════════════
# Comprehensive Evaluation
# ═══════════════════════════════════════════════

# Load best model
ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "best_model.pth")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(CONFIG["local_backup_dir"], "best_model.pth")

ckpt = torch.load(ckpt_path, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['epoch']} (PSNR: {ckpt['val_psnr']:.2f}, SSIM: {ckpt['val_ssim']:.4f})")

eval_configs = [
    ("Gaussian 0.05", "gaussian", 0.05),
    ("Gaussian 0.10", "gaussian", 0.10),
    ("Gaussian 0.15", "gaussian", 0.15),
    ("Gaussian 0.20", "gaussian", 0.20),
    ("Rician 0.05",   "rician",   0.05),
    ("Rician 0.10",   "rician",   0.10),
    ("S&P 1%",        "salt_pepper", 0.01),
    ("S&P 3%",        "salt_pepper", 0.03),
    ("S&P 5%",        "salt_pepper", 0.05),
    ("Poisson 30",    "poisson",  30.0),
    ("Poisson 50",    "poisson",  50.0),
    ("Mixed (G+SP)",  "mixed",    0.05),
]

print("\n" + "=" * 90)
print(f"  {'Noise Type':>22} | {'PSNR Noisy':>12} | {'PSNR Denoised':>14} | {'SSIM':>8} | {'Delta PSNR':>10}")
print("=" * 90)

all_results = []
model.eval()

for name, noise_type, level in eval_configs:
    noise_fn = NOISE_FNS[noise_type]
    noise_idx_val = NOISE_TYPE_MAP[noise_type]
    psnr_noisy, psnr_denoised, ssim_vals = [], [], []

    with torch.no_grad():
        for noisy, clean, _ in val_loaders['gaussian']:  # Use any val loader for images
            clean_dev = clean.to(DEVICE)
            # Apply specific noise
            noisy_fixed = torch.stack([noise_fn(c, level) for c in clean]).to(DEVICE)
            noise_idx_tensor = torch.full((clean.shape[0],), noise_idx_val, dtype=torch.long, device=DEVICE)

            denoised = model(noisy_fixed, noise_idx_tensor)

            for i in range(clean.shape[0]):
                c = clean[i,0].numpy()
                n = noisy_fixed[i,0].cpu().numpy()
                d = denoised[i,0].cpu().numpy()
                psnr_noisy.append(sk_psnr(c, n, data_range=1.0))
                psnr_denoised.append(sk_psnr(c, d, data_range=1.0))
                ssim_vals.append(sk_ssim(c, d, data_range=1.0))

    pn = np.mean(psnr_noisy)
    pd = np.mean(psnr_denoised)
    sd = np.mean(ssim_vals)
    delta = pd - pn

    print(f"  {name:>22} | {pn:>11.2f} | {pd:>13.2f} | {sd:>7.4f} | {delta:>+9.2f}")
    all_results.append({'name': name, 'noise_type': noise_type, 'level': level,
                        'psnr_noisy': pn, 'psnr_denoised': pd, 'ssim': sd, 'delta_psnr': delta})

print("=" * 90)

Loaded best model from epoch 100 (PSNR: 35.67, SSIM: 0.9045)

              Noise Type |   PSNR Noisy |  PSNR Denoised |     SSIM | Delta PSNR
           Gaussian 0.05 |       27.01 |         36.06 |  0.9512 |     +9.05
           Gaussian 0.10 |       21.20 |         32.78 |  0.9156 |    +11.58
           Gaussian 0.15 |       17.88 |         30.93 |  0.8876 |    +13.05
           Gaussian 0.20 |       15.62 |         29.62 |  0.8624 |    +14.01
             Rician 0.05 |       25.17 |         35.80 |  0.9414 |    +10.64
             Rician 0.10 |       18.96 |         32.35 |  0.9004 |    +13.39
                  S&P 1% |       24.14 |         54.55 |  0.9993 |    +30.41
                  S&P 3% |       19.40 |         51.33 |  0.9987 |    +31.93
                  S&P 5% |       17.22 |         49.39 |  0.9981 |    +32.16
              Poisson 30 |       22.40 |         27.00 |  0.8732 |     +4.60
              Poisson 50 |       24.59 |         27.93 |  0.8932 |     +3.34
          

---
## Step 12: Visual Comparison Grid

In [13]:
model.eval()
fig, axes = plt.subplots(5, 3, figsize=(14, 22))
for ax, col in zip(axes[0], ["Noisy Input", "Denoised (Model)", "Clean Ground Truth"]):
    ax.set_title(col, fontsize=14, fontweight="bold")

vis_configs = [
    ("Gaussian 0.10", "gaussian", 0.10),
    ("Rician 0.08",   "rician",   0.08),
    ("S&P 2%",        "salt_pepper", 0.02),
    ("Poisson 30",    "poisson",  30.0),
    ("Mixed (G+SP)",  "mixed",    0.05),
]

# Get one clean image
sample_clean = None
for noisy, clean, _ in val_loaders['gaussian']:
    sample_clean = clean[0:1]
    break

for row, (label, nt, level) in enumerate(vis_configs):
    noise_fn = NOISE_FNS[nt]
    ci = sample_clean[0]
    ni = noise_fn(ci, level)
    noise_idx_tensor = torch.tensor([NOISE_TYPE_MAP[nt]], device=DEVICE)

    with torch.no_grad():
        di = model(ni.unsqueeze(0).to(DEVICE), noise_idx_tensor).cpu().squeeze()

    c_np, n_np, d_np = ci.squeeze().numpy(), ni.squeeze().numpy(), di.numpy()
    pn = sk_psnr(c_np, n_np, data_range=1.0)
    pd = sk_psnr(c_np, d_np, data_range=1.0)
    sd = sk_ssim(c_np, d_np, data_range=1.0)

    axes[row][0].imshow(n_np, cmap="gray"); axes[row][0].axis("off")
    axes[row][0].set_ylabel(label, fontsize=11, rotation=0, labelpad=80)
    axes[row][1].imshow(d_np, cmap="gray"); axes[row][1].axis("off")
    axes[row][1].text(5, 15, f"PSNR: {pd:.1f}dB\nSSIM: {sd:.3f}", color="lime",
                      fontsize=9, bbox=dict(boxstyle="round", facecolor="black", alpha=0.7))
    axes[row][2].imshow(c_np, cmap="gray"); axes[row][2].axis("off")

plt.suptitle("Medical Image Denoising — Brain Tumor MRI\nResUNet + Attention + Noise Conditioning + Curriculum",
             fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("denoising_results_all_noise.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Step 13: Metrics Charts

In [14]:
noise_groups = {}
for r in all_results:
    nt = r['noise_type']
    if nt not in noise_groups:
        noise_groups[nt] = {'names': [], 'psnr_noisy': [], 'psnr_denoised': [], 'ssim': [], 'delta': []}
    noise_groups[nt]['names'].append(r['name'])
    noise_groups[nt]['psnr_noisy'].append(r['psnr_noisy'])
    noise_groups[nt]['psnr_denoised'].append(r['psnr_denoised'])
    noise_groups[nt]['ssim'].append(r['ssim'])
    noise_groups[nt]['delta'].append(r['delta_psnr'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gaussian PSNR across sigma
if 'gaussian' in noise_groups:
    g = noise_groups['gaussian']
    sigmas = [0.05, 0.10, 0.15, 0.20]
    axes[0].plot(sigmas, g['psnr_noisy'], "r--o", label="Noisy", linewidth=2)
    axes[0].plot(sigmas, g['psnr_denoised'], "g-o", label="Denoised", linewidth=2)
    axes[0].set_xlabel("sigma"); axes[0].set_ylabel("PSNR (dB)")
    axes[0].set_title("PSNR: Gaussian Noise", fontweight='bold')
    axes[0].legend(loc='best'); axes[0].grid(True, alpha=0.3)

# PSNR improvement by noise type
nt_names, deltas, ssim_vals = [], [], []
for nt, data in noise_groups.items():
    nt_names.append(nt.replace('_', '\n'))
    deltas.append(np.mean(data['delta']))
    ssim_vals.append(np.mean(data['ssim']))

colors = ['#ff6b6b', '#51cf66', '#74c0fc', '#ffd43b', '#cc5de8']
axes[1].bar(range(len(nt_names)), deltas, color=colors[:len(nt_names)])
axes[1].set_xticks(range(len(nt_names))); axes[1].set_xticklabels(nt_names, fontsize=9)
axes[1].set_ylabel("PSNR Improvement (dB)"); axes[1].set_title("Delta PSNR by Noise Type", fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].bar(range(len(nt_names)), ssim_vals, color=colors[:len(nt_names)])
axes[2].set_xticks(range(len(nt_names))); axes[2].set_xticklabels(nt_names, fontsize=9)
axes[2].set_ylabel("SSIM"); axes[2].set_title("Denoised SSIM by Noise Type", fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Evaluation Metrics — ResUNet + Noise Conditioning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("metrics_charts.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Step 14: Save Final Model

In [15]:
# Save final model
final_local = os.path.join(CONFIG["local_backup_dir"], "final_model.pth")
torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "test_results": all_results,
    "training_history": history,
}, final_local)
print(f"Final model saved: {final_local} ({os.path.getsize(final_local)/1e6:.1f} MB)")

# Sync to Drive
try:
    shutil.copy2(final_local, os.path.join(CONFIG["checkpoint_dir"], "final_model.pth"))
    print("Synced to Drive")
except Exception as e:
    print(f"Drive sync error: {e}")

# Save history JSON
hist_path = os.path.join(CONFIG["local_backup_dir"], "training_history.json")
with open(hist_path, 'w') as f:
    json.dump(history, f, indent=2, default=str)
print(f"History saved: {hist_path}")

print("\nASSIGNMENT COMPLETE!")
print("  - Model: ResUNet + SE Attention + Noise Conditioning")
print("  - Training: Curriculum (3 phases) + Residual Learning")
print("  - Loss: Charbonnier + SSIM + L1 + Edge (rebalanced)")
print("  - Evaluated: PSNR & SSIM across 5 noise types at multiple levels")

Final model saved: /kaggle/working/final_model.pth (130.7 MB)
Drive sync error: '/kaggle/working/final_model.pth' and '/kaggle/working/final_model.pth' are the same file
History saved: /kaggle/working/training_history.json

ASSIGNMENT COMPLETE!
  - Model: ResUNet + SE Attention + Noise Conditioning
  - Training: Curriculum (3 phases) + Residual Learning
  - Loss: Charbonnier + SSIM + L1 + Edge (rebalanced)
  - Evaluated: PSNR & SSIM across 5 noise types at multiple levels


---
## Step 15: Denoise Your Own Images

Upload any MRI/CT scan and denoise it. You specify the noise type,
or let the model try to auto-detect the best one.

In [14]:
# ═══════════════════════════════════════════════
# Step 15: Denoise Your Own Images
# ═══════════════════════════════════════════════

# ── Load trained model weights ──
MODEL_PATH = "/kaggle/input/datasets/camorednex/brain-mri-best-model/best_model_medDenoise.pth"
state = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(state if "model_state_dict" not in state else state["model_state_dict"])
model.eval()
print(f"Model loaded from {MODEL_PATH}")

# ── Denoise function ──
def denoise_image(model, image_path, noise_type='gaussian', image_size=256, device=DEVICE):
    """Denoise an uploaded image with a specified noise type assumption.
    
    Args:
        noise_type: 'gaussian', 'rician', 'salt_pepper', 'poisson', or 'mixed'
    """
    image = Image.open(image_path).convert("L")
    original_size = image.size

    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])

    img_tensor = transform(image).to(device)
    noise_idx = torch.tensor([NOISE_TYPE_MAP[noise_type]], device=device)

    model.eval()
    with torch.no_grad():
        denoised = model(img_tensor.unsqueeze(0), noise_idx).cpu()

    denoised_pil = TF.to_pil_image(denoised.squeeze())
    denoised_pil = denoised_pil.resize(original_size, Image.BILINEAR)

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    o_np = img_tensor.cpu().squeeze().numpy()
    d_np = denoised.squeeze().numpy()
    diff = np.clip(np.abs(o_np - d_np) * 5, 0, 1)  # 5x amplified

    axes[0].imshow(o_np, cmap='gray'); axes[0].axis('off')
    axes[0].set_title('Your Image', fontsize=14, fontweight='bold')
    axes[1].imshow(d_np, cmap='gray'); axes[1].axis('off')
    axes[1].set_title(f'Denoised ({noise_type})', fontsize=14, fontweight='bold')
    axes[2].imshow(diff, cmap='hot'); axes[2].axis('off')
    axes[2].set_title('What Changed (5x)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    return denoised_pil

print("Ready! Use denoise_image(model, 'path.jpg', noise_type='gaussian')")

Model loaded from /kaggle/input/datasets/camorednex/brain-mri-best-model/best_model_medDenoise.pth
Ready! Use denoise_image(model, 'path.jpg', noise_type='gaussian')


In [28]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage
import io

model.eval()

upload = widgets.FileUpload(accept='image/*', multiple=True, description='Upload MRI/CT Scans')
btn = widgets.Button(description='Denoise', button_style='success')
out = widgets.Output()

def on_btn(b):
    with out:
        clear_output(wait=True)
        if not upload.value:
            print("Upload images first!")
            return
        preprocess = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
        ])
        for f in upload.value:
            filename = f['name']
            print(f"Processing: {filename}")
            img = Image.open(io.BytesIO(f['content'])).convert("L")
            input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)
            
            # Try all 5 noise types, show ALL results
            input_np = input_tensor.cpu().squeeze().numpy()
            results = []
            for i in range(5):
                with torch.no_grad():
                    d = model(input_tensor, torch.tensor([i], device=DEVICE)).cpu().squeeze().numpy()
                results.append(d.clip(0,1))
            
            # Show: Original + all 5 denoised
            fig, ax = plt.subplots(2, 3, figsize=(15, 10))
            names = ['Original', 'Gaussian', 'Rician', 'Salt & Pepper', 'Poisson', 'Mixed']
            ax[0,0].imshow(input_np, cmap='gray'); ax[0,0].set_title(names[0], fontweight='bold'); ax[0,0].axis('off')
            for i in range(5):
                r, c = (i+1)//3, (i+1)%3
                ax[r,c].imshow(results[i], cmap='gray'); ax[r,c].set_title(names[i+1], fontweight='bold'); ax[r,c].axis('off')
            plt.suptitle(filename, fontweight='bold')
            plt.tight_layout()
            
            save_path = f"/kaggle/working/{filename.rsplit('.',1)[0]}_result.png"
            plt.savefig(save_path, dpi=100, bbox_inches='tight')
            plt.close()
            display(IPImage(filename=save_path))

btn.on_click(on_btn)
display(widgets.VBox([upload, btn, out]))

---
## Step 16: Verify Saved Outputs


In [18]:
# ═══════════════════════════════════════════════
# Verify all saved outputs
# ═══════════════════════════════════════════════

print('All outputs saved in /kaggle/working/')
print('Contents:')
for f in sorted(os.listdir('/kaggle/working/')):
    fpath = os.path.join('/kaggle/working/', f)
    if os.path.isfile(fpath):
        print(f'  {f}: {os.path.getsize(fpath)/1e6:.1f} MB')
    elif os.path.isdir(fpath):
        count = len(os.listdir(fpath))
        print(f'  {f}/ ({count} files)')

print('\nTo download outputs from Kaggle:')
print('  1. Click the Output tab at the bottom of the notebook')
print('  2. Or find files in the right panel under Output')


All outputs saved in /kaggle/working/
Contents:
  __notebook__.ipynb: 0.2 MB
  best_model.pth: 392.1 MB
  checkpoint_epoch_100.pth: 392.1 MB
  checkpoint_epoch_40.pth: 392.1 MB
  checkpoint_epoch_50.pth: 392.1 MB
  checkpoint_epoch_60.pth: 392.1 MB
  checkpoint_epoch_70.pth: 392.1 MB
  checkpoint_epoch_80.pth: 392.1 MB
  checkpoint_epoch_90.pth: 392.1 MB
  denoise_comparison.png: 2.9 MB
  denoising_results_all_noise.png: 3.4 MB
  final_model.pth: 130.7 MB
  metrics_charts.png: 0.1 MB
  training_history.json: 0.0 MB
  training_history.png: 0.3 MB
  wandb/ (4 files)

To download outputs from Kaggle:
  1. Click the Output tab at the bottom of the notebook
  2. Or find files in the right panel under Output
